# Data
[Please click here to get the data](https://drive.google.com/file/d/1kxtGNMN-U-uVvAQZRkd-k-8zZG8-LC6r/view?usp=sharing)

### Explanation of the Demand Forecasting

This code block performs demand forecasting for a 'dark store' using historical Instacart order data and an XGBoost machine learning model. The output is a CSV file (`dark_store_demand.csv`) containing both historical and predicted hourly demand data, suitable for a Power BI dashboard.

#### Situation
The initial situation involves having a dataset (`orders.csv`) from Instacart, which contains historical order information, but lacks real-world dates, only providing the day of the week (`order_dow`) and hour of the day (`order_hour_of_day`). The goal is to build a robust demand forecasting system.

#### Task
The primary task is to develop a demand forecasting solution that can leverage the provided Instacart data to predict future demand. This involves several sub-tasks:
1.  **Prepare and Engineer Features**: Transform the raw Instacart data into a time-series format suitable for machine learning, including generating a synthetic timeline and creating relevant features (e.g., hour, day of week, weekend indicator).
2.  **Train a Machine Learning Model**: Utilize the prepared historical data to train a predictive model capable of recognizing and forecasting demand patterns.
3.  **Generate Future Predictions**: Use the trained model to forecast demand for a future period.
4.  **Export Data**: Combine historical and predicted data into a single, easily consumable format (CSV) for use in analytics tools like Power BI.

#### Action
Here's a breakdown of the actions taken:
1.  **Data Loading**: The `orders.csv` file was loaded into a pandas DataFrame.
2.  **Demand Aggregation**: The code grouped the `df` by `order_dow` and `order_hour_of_day` to calculate the `total_orders` for each combination, establishing a baseline for typical demand patterns.
3.  **Synthetic Historical Data Generation**: A 30-day historical timeline, ending on the current date, was synthetically generated. For each day in this range, the aggregated demand patterns were applied, with a small random noise added to simulate real-world variability. This resulted in the `history_df` DataFrame, capturing historical hourly demand.
4.  **Feature Selection for Training**: The `Hour_of_Day`, `Day_of_Week`, and `Is_Weekend` columns from `history_df` were selected as input features (`X_train`), while `Demand` was set as the target variable (`y_train`).
5.  **Model Training (XGBoost)**: An `XGBRegressor` model was initialized and trained on `X_train` and `y_train` to learn the relationships between time-based features and demand.
6.  **Future Demand Prediction**: A 7-day future period, starting from the day after the synthetic historical period ends, was defined. For each hour within these future days, corresponding features were created, and the trained XGBoost model was used to predict hourly demand. These predictions were compiled into the `future_df` DataFrame.
7.  **Data Combination and Export**: Finally, `history_df` and `future_df` were concatenated to create a comprehensive `final_dataset`, which was then saved as `dark_store_demand.csv`. This CSV includes both the synthetic historical data and the machine learning-generated future predictions.

#### Result
The execution of the code resulted in:
1.  **A Trained Forecasting Model**: A functional XGBoost model capable of predicting hourly demand based on time-related features.
2.  **A Comprehensive Data Export**: A CSV file named `dark_store_demand.csv` was successfully generated. This file contains a consolidated view of 30 days of synthetic historical hourly demand data and 7 days of forecasted hourly demand data.
3.  **Dashboard-Ready Data**: The `dark_store_demand.csv` file is now ready for download from the Colab environment and can be directly loaded into a Power BI dashboard (or any other visualization tool) to display historical demand trends and future predictions, aiding in operational planning and decision-making.

===========================================================================================================================================

In [1]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from datetime import datetime, timedelta

print("Loading Instacart data...")
# Load the dataset (Make sure orders.csv is uploaded to your Colab files)
df = pd.read_csv('orders.csv')

# 1. DATA PREPARATION & FEATURE ENGINEERING
print("Processing historical data...")
# We only need the day of week and hour for demand forecasting
demand_df = df.groupby(['order_dow', 'order_hour_of_day']).size().reset_index(name='total_orders')

# The Instacart data doesn't have real dates. Let's generate a synthetic 30-day historical timeline
# ending today, so your Power BI dashboard looks like a live, real-world system.
end_date = datetime.now().date()
start_date = end_date - timedelta(days=30)
date_range = pd.date_range(start=start_date, end=end_date, freq='D')

historical_data = []
for date in date_range:
    # Match the calendar day of the week (0=Monday, 6=Sunday) to Instacart's 'order_dow'
    dow = date.weekday()
    daily_demand = demand_df[demand_df['order_dow'] == dow].copy()

    for _, row in daily_demand.iterrows():
        # Add some slight random noise so the days don't look identical in Power BI
        noise = np.random.randint(-15, 15)
        orders = max(0, row['total_orders'] + noise)

        historical_data.append({
            'Timestamp': datetime.combine(date, datetime.min.time()) + timedelta(hours=row['order_hour_of_day']),
            'Hour_of_Day': row['order_hour_of_day'],
            'Day_of_Week': dow,
            'Is_Weekend': 1 if dow >= 5 else 0,
            'Demand': orders,
            'Data_Type': 'Historical'
        })

history_df = pd.DataFrame(historical_data)

# 2. MACHINE LEARNING: TRAINING THE MODEL
print("Training the XGBoost forecasting model...")
# Define features (X) and target variable (y)
X_train = history_df[['Hour_of_Day', 'Day_of_Week', 'Is_Weekend']]
y_train = history_df['Demand']

# Initialize and train the XGBoost Regressor
model = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# 3. GENERATING FUTURE PREDICTIONS
print("Predicting demand for the next 7 days...")
future_dates = pd.date_range(start=end_date + timedelta(days=1), periods=7, freq='D')
predicted_data = []

for date in future_dates:
    dow = date.weekday()
    for hour in range(24):
        # Create features for the future hour
        future_features = pd.DataFrame({
            'Hour_of_Day': [hour],
            'Day_of_Week': [dow],
            'Is_Weekend': [1 if dow >= 5 else 0]
        })

        # Predict the demand
        predicted_orders = int(model.predict(future_features)[0])

        predicted_data.append({
            'Timestamp': datetime.combine(date, datetime.min.time()) + timedelta(hours=hour),
            'Hour_of_Day': hour,
            'Day_of_Week': dow,
            'Is_Weekend': 1 if dow >= 5 else 0,
            'Demand': max(0, predicted_orders), # Ensure no negative predictions
            'Data_Type': 'Predicted'
        })

future_df = pd.DataFrame(predicted_data)

# 4. EXPORTING FOR POWER BI
print("Combining and exporting data...")
# Combine historical and predicted data into one clean dataset
final_dataset = pd.concat([history_df, future_df], ignore_index=True)

# Save to CSV
output_filename = 'dark_store_demand.csv'
final_dataset.to_csv(output_filename, index=False)
print(f"Success! Download '{output_filename}' from the Colab file explorer and load it into Power BI.")

Loading Instacart data...
Processing historical data...
Training the XGBoost forecasting model...
Predicting demand for the next 7 days...
Combining and exporting data...
Success! Download 'dark_store_demand.csv' from the Colab file explorer and load it into Power BI.


============================================================================================================================================

# ***Guddu Kumar Mishra***